In [3]:
import os, json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from src import Questions
import matplotlib.patches as mpatches

In [4]:
# definitions
models = [
    "gpt_4o",
    "llama31_70B_Inst",
    "Qwen25_72B_Inst",
    "llama31_8B_Inst",
    "Qwen25_7B_Inst",
    "llama32_1B_Inst",
    "Qwen25_1dot5B_Inst"
]
task_types = [
    'mt_and_summarization',
    'syntaxes',
    'knowledge_and_reasoning',
    'semantics',
    'qa_extraction'
]
demography_profiles = [
    "employment_sector",
    "race",
    "education",
    "sex",
    "age",
    "language",
    "nationality_group"
]
SCORED_ANNOTATOR_DIR = "annotation_artefacts/final_annotator_objects"

In [5]:
# creating a dataframe of all of the annotator profiles
questions = Questions().get_questions()

In [6]:
def calculate_diversity_score(list_of_values):
    if len(list_of_values) < 2:
        return 0  # No diversity if less than 2 elements
    
    total = 0
    count = 0
    for i in range(len(list_of_values)):
        for j in range(i + 1, len(list_of_values)):
            total += abs(list_of_values[i] - list_of_values[j])
            count += 1
    return total / count

model_diversity_scores = []
for model in models:
    model_specific_scores = []
    for annfilename in os.listdir(SCORED_ANNOTATOR_DIR):
        annobj_for_df = {}
        annotator_filepath = os.path.join(SCORED_ANNOTATOR_DIR, annfilename)
        with open(annotator_filepath, "r") as ao:
            annotator_object = json.load(ao)
        
        annobj_for_df["id"] = annotator_object["id"]
        annobj_for_df.update(**annotator_object["profile"])
        
        for i, (question, scores_for_question) in enumerate(annotator_object["answers"].items()):
            if not scores_for_question["answer"]:
                continue
            else:
                for qdi, scores in scores_for_question["downstream_scores"][model].items():
                    annobj_for_df[qdi] = list(scores.values())[0]
        model_specific_scores.append(annobj_for_df)
    score_df = pd.DataFrame(model_specific_scores)
    exclude_cols = ["id"] + demography_profiles
    task_cols = [col for col in score_df.columns if col not in exclude_cols]
    score_df[task_cols] = (score_df[task_cols] - score_df[task_cols].min()) / (score_df[task_cols].max() - score_df[task_cols].min())
    
    demography_profile_means = {}
    for dp in demography_profiles:
        demo_avgs = {}
        demo_grouped = score_df.groupby([dp])
        for demo_name, group in demo_grouped:
            # implement here for exceptions
            # here
            # implement here for exceptions
            task_avgs = []
            for tc in task_cols:
                task_avgs.append(group[tc].mean())
            demo_avgs[demo_name[0]] = np.nanmean(task_avgs)
        demography_profile_means[dp] = demo_avgs
    
    diversity_scores = {}
    for dpm_k, dpm_v in demography_profile_means.items():
        diversity_scores[dpm_k] = calculate_diversity_score(list(dpm_v.values()))
    
    model_diversity_scores.append({
        "model" : model,
        **diversity_scores
    })
            

In [7]:
model_diversity_df = pd.DataFrame(model_diversity_scores)
model_diversity_df

,model,employment_sector,race,education,sex,age,language,nationality_group
0,gpt_4o,0.032555,0.023181,0.012563,0.017490,0.019172,0.036781,0.025735
1,llama31_70B_Inst,0.028382,0.019987,0.013151,0.001440,0.017707,0.025899,0.025332
2,Qwen25_72B_Inst,0.031143,0.015809,0.014220,0.002118,0.023527,0.022264,0.025516
3,llama31_8B_Inst,0.027906,0.008379,0.025363,0.006987,0.017343,0.010427,0.008256
4,Qwen25_7B_Inst,0.021677,0.019841,0.011355,0.009000,0.017525,0.025837,0.024499
5,llama32_1B_Inst,0.036830,0.030927,0.021726,0.010996,0.020318,0.021244,0.014881
6,Qwen25_1dot5B_Inst,0.012976,0.009609,0.011464,0.009069,0.013720,0.012186,0.011235


In [64]:
diversity_scores

{'employment_sector': 0.032554564491711645,
 'race': 0.023180707137355094,
 'education': 0.012563055738997667,
 'sex': 0.017490004120560676,
 'age': 0.019171677876993403,
 'language': 0.03678104324749776,
 'nationality_group': 0.02573519036539923}

In [43]:
for name, group in aaa:
    print(f"Group: {name}")
    break
    # print(group)


Group: ('Engineering-STEM',)


In [44]:
group

,id,employment_sector,race,education,sex,age,language,nationality_group,0_0,1_0,...,27_5,27_4,27_3,27_2,27_1,27_0,5_0,7_0,10_0,28_0
1,63,Engineering-STEM,White,Postgraduate,male,31-40,L1,Europe,0.512191,0.538021,...,0.699922,0.808592,0.774446,0.635017,0.871573,0.778577,1.0,1.0,0.750,0.35
3,22,Engineering-STEM,Black,Postgraduate,female,18-30,L1,Africa,0.621525,0.629431,...,0.660315,0.666339,0.822756,0.680339,0.664472,0.717848,1.0,1.0,0.750,0.50
5,18,Engineering-STEM,White,Graduate,female,41-50,L1,Europe,NaN,0.333913,...,0.671079,0.581767,0.995523,0.614050,0.600908,0.640602,1.0,0.5,0.750,NaN
6,38,Engineering-STEM,Asian,Graduate,male,18-30,L2,Asia,0.597383,0.760058,...,0.817923,0.860204,0.898956,0.875327,0.940660,0.869517,1.0,1.0,0.750,0.10
8,96,Engineering-STEM,White,Graduate,male,41-50,L1,Europe,0.516942,0.574918,...,0.774867,0.793128,0.781254,0.824634,0.829149,0.892016,1.0,NaN,0.750,0.35
11,55,Engineering-STEM,Asian,Graduate,female,18-30,L2,Asia,0.504930,0.575790,...,0.629960,0.636562,0.963451,0.649302,0.611022,0.610308,0.0,0.5,0.500,0.35
12,104,Engineering-STEM,White,Graduate,male,41-50,L1,North America,0.728191,0.482858,...,0.590287,0.664629,0.735463,0.655829,0.673992,0.645223,1.0,0.5,0.750,0.50
13,43,Engineering-STEM,Asian,Graduate,male,18-30,L2,Asia,0.483770,0.685875,...,1.000000,0.939778,0.954481,0.985663,0.998300,0.986087,1.0,0.5,0.750,0.70
17,54,Engineering-STEM,Asian,Highschool,male,18-30,L2,Asia,0.526219,0.686351,...,0.968912,0.967824,0.996374,0.992127,0.952258,1.000000,1.0,0.5,0.750,0.65
20,3,Engineering-STEM,White,Graduate,female,31-40,L1,Europe,0.369251,0.630950,...,0.195787,0.143634,0.339685,0.207002,0.154078,0.310295,0.0,1.0,0.750,0.15


In [23]:
score_df

,id,employment_sector,race,education,sex,age,language,nationality_group,0_0,1_0,...,27_5,27_4,27_3,27_2,27_1,27_0,5_0,7_0,10_0,28_0
0,34,Humanities,Black,Postgraduate,female,31-40,None,North America,0.659508,0.907711,...,0.749596,0.771669,0.833510,0.764036,0.772879,0.756421,NaN,NaN,NaN,NaN
1,63,Engineering-STEM,White,Postgraduate,male,31-40,L1,Europe,0.611266,0.913748,...,0.768058,0.830816,0.757244,0.762395,0.858661,0.785321,1.0,1.0,0.90,0.9475
2,75,Health-STEM,White,Graduate,female,31-40,None,Europe,0.593466,0.921699,...,0.823164,0.870755,0.773587,0.825062,0.859093,0.784954,0.0,0.9,0.90,0.9600
3,22,Engineering-STEM,Black,Postgraduate,female,18-30,L1,Africa,0.639355,0.923687,...,0.755007,0.777103,0.776563,0.778827,0.774977,0.764861,1.0,1.0,0.90,0.9550
4,59,Health-STEM,White,Graduate,female,31-40,L1,Europe,NaN,0.907668,...,0.558510,0.546982,0.488149,0.556858,0.560086,0.546212,1.0,0.9,0.90,0.9625
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,37,Humanities,Asian,Highschool,female,18-30,L2,Asia,0.671164,0.945885,...,0.785565,0.868439,0.798007,0.809454,0.878266,0.815200,1.0,0.9,0.95,0.9375
95,60,Health-STEM,Asian,Postgraduate,male,41-50,L1,Europe,0.604593,NaN,...,0.653765,0.644509,0.686018,0.650850,0.621912,0.654623,1.0,NaN,0.90,NaN
96,76,Engineering-STEM,White,Postgraduate,male,31-40,None,Africa,0.629583,0.927773,...,0.665354,0.663794,0.702836,0.683152,0.681333,0.680275,NaN,1.0,0.90,0.9625
97,99,Engineering-STEM,Black,Postgraduate,female,41-50,None,North America,0.641109,0.899479,...,0.825610,0.882274,0.835826,0.868134,0.867274,0.820729,1.0,NaN,0.95,0.9725
